In [0]:

from pyspark.sql import functions as F

# Ler a Silver
clientes = spark.table("first_catalog.default.silver_clientes")

# Criar dimensão de clientes
dim_cliente = clientes.select(
    "id_cliente",
    "nome_cliente",
    "segmento",
    "cidade",
    "estado",
    "pais",
    "mercado",
    "regiao"
)

# Validar a chave da dimensão
duplicados = (
    dim_cliente
    .groupBy("id_cliente")
    .count()
    .filter(F.col("count") > 1)
)

print("Total de clientes:", dim_cliente.count())
print("IDs duplicados:", duplicados.count())

# Guardar como tabela Gold
(
    dim_cliente.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("first_catalog.default.dim_cliente")
)

display(dim_cliente.limit(10))

Total de clientes: 1590
IDs duplicados: 0


id_cliente,nome_cliente,segmento,cidade,estado,pais,mercado,regiao
RH-19495,Rick Hansen,Consumidor,New York City,New York,United States,US,East
JR-16210,Justin Ritter,Corporativo,Wollongong,New South Wales,Australia,APAC,Oceania
CR-12730,Craig Reiter,Consumidor,Brisbane,Queensland,Australia,APAC,Oceania
KM-16375,Katherine Murray,Outros,Berlin,Berlin,Germany,EU,Central
RH-9495,Rick Hansen,Consumidor,Dakar,Dakar,Senegal,Africa,Africa
JM-15655,Jim Mitchum,Corporativo,Sydney,New South Wales,Australia,APAC,Oceania
TS-21340,Toby Swindell,Consumidor,Porirua,Wellington,New Zealand,APAC,Oceania
MB-18085,Mick Brown,Consumidor,Hamilton,Waikato,New Zealand,APAC,Oceania
JW-15220,Jane Waco,Corporativo,Sacramento,California,United States,US,West
JH-15985,Joseph Holt,Consumidor,Concord,North Carolina,United States,US,South


In [0]:

produtos = spark.table("first_catalog.default.silver_produtos")

dim_produto = produtos.select(
    "id_produto",
    "categoria",
    "subcategoria",
    "nome_produto"
)

duplicados = (
    dim_produto
    .groupBy("id_produto")
    .count()
    .filter(F.col("count") > 1)
)

print("Total de produtos:", dim_produto.count())
print("IDs duplicados:", duplicados.count())

(
    dim_produto.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("first_catalog.default.dim_produto")
)

display(dim_produto.limit(10))

Total de produtos: 10292
IDs duplicados: 0


id_produto,categoria,subcategoria,nome_produto
FUR-CH-10003950,Moveis,Chairs,"Novimex Executive Leather Armchair, Black"
OFF-AP-10004512,Material de Escritorio,Appliances,"Hoover Stove, Red"
TEC-MA-10004125,Tecnologia,Machines,Cubify CubeX 3D Printer Triple Head Print
FUR-BO-10001372,Moveis,Bookcases,"Safco Classic Bookcase, Pine"
TEC-MOT-10002272,Tecnologia,Phones,"Motorola Smart Phone, with Caller ID"
TEC-CO-10001766,Tecnologia,Copiers,Canon PC940 Copier
FUR-BO-10004999,Moveis,Bookcases,"Safco Classic Bookcase, Metal"
OFF-HOO-10001881,Material de Escritorio,Appliances,"Hoover Stove, White"
TEC-MOT-10000554,Tecnologia,Phones,"Motorola Smart Phone, Full Size"
FUR-TA-10004575,Moveis,Tables,Hon 5100 Series Wood Tables


In [0]:

from pyspark.sql import functions as F

pedidos = spark.table("first_catalog.default.silver_pedidos")

dim_pedido = pedidos.select(
    "id_pedido",
    "data_pedido",
    "data_envio",
    "modo_envio",
    "prioridade_pedido",
    "dias_para_envio"
)

# Validar unicidade da chave
duplicados = (
    dim_pedido
    .groupBy("id_pedido")
    .count()
    .filter(F.col("count") > 1)
)

print("Total de pedidos:", dim_pedido.count())
print("IDs de pedido duplicados:", duplicados.count())

(
    dim_pedido.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("first_catalog.default.dim_pedido")
)

display(dim_pedido.limit(10))

Total de pedidos: 25035
IDs de pedido duplicados: 0


id_pedido,data_pedido,data_envio,modo_envio,prioridade_pedido,dias_para_envio
ES-2014-5099955,2014-07-31,2014-08-03,Segunda Classe,Critico,3
CG-2011-8610,2011-09-14,2011-09-15,Primeira Classe,Alto,1
US-2014-168116,2014-11-05,2014-11-05,Mesmo Dia,Alto,0
MX-2011-110275,2011-11-22,2011-11-23,Primeira Classe,Medio,1
CA-2012-145352,2012-03-16,2012-03-22,Classe Padrao,Medio,6
ES-2014-3785216,2014-11-19,2014-11-23,Classe Padrao,Alto,4
IT-2012-1779015,2012-09-29,2012-09-29,Mesmo Dia,Critico,0
ES-2012-4765389,2012-12-26,2012-12-28,Segunda Classe,Critico,2
IN-2014-39077,2014-07-19,2014-07-23,Classe Padrao,Alto,4
SO-2014-4990,2014-09-18,2014-09-20,Primeira Classe,Alto,2


### Criar a dimensão de datas
A dimensão de datas permite agrupar vendas por ano, trimestre, mês, dia da semana e outros períodos, sem ter de repetir essas expressões em cada consulta.

In [0]:

pedidos = spark.table("first_catalog.default.silver_pedidos")

dim_data = (
    pedidos
    .select(F.col("data_pedido").alias("data"))
    .distinct()
    .withColumn("ano", F.year("data"))
    .withColumn("trimestre", F.quarter("data"))
    .withColumn("mes", F.month("data"))
    .withColumn("nome_mes", F.date_format("data", "MMMM"))
    .withColumn("dia", F.dayofmonth("data"))
    .withColumn("dia_semana_numero", F.dayofweek("data"))
    .withColumn("dia_semana", F.date_format("data", "EEEE"))
    .withColumn(
        "fim_de_semana",
        F.dayofweek("data").isin([1, 7])
    )
)

print("Datas distintas:", dim_data.count())
print("Datas nulas:", dim_data.filter(F.col("data").isNull()).count())

(
    dim_data.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("first_catalog.default.dim_data")
)

display(dim_data.orderBy("data").limit(10))

Datas distintas: 1428
Datas nulas: 0


data,ano,trimestre,mes,nome_mes,dia,dia_semana_numero,dia_semana,fim_de_semana
2011-01-01,2011,1,1,January,1,7,Saturday,true
2011-01-02,2011,1,1,January,2,1,Sunday,true
2011-01-03,2011,1,1,January,3,2,Monday,false
2011-01-04,2011,1,1,January,4,3,Tuesday,false
2011-01-05,2011,1,1,January,5,4,Wednesday,false
2011-01-06,2011,1,1,January,6,5,Thursday,false
2011-01-07,2011,1,1,January,7,6,Friday,false
2011-01-08,2011,1,1,January,8,7,Saturday,true
2011-01-09,2011,1,1,January,9,1,Sunday,true
2011-01-10,2011,1,1,January,10,2,Monday,false


Construir a tabela de factos

In [0]:

from pyspark.sql import functions as F

# Ler as tabelas Silver
vendas = spark.table("first_catalog.default.silver_vendas")
clientes = spark.table("first_catalog.default.silver_clientes")
produtos = spark.table("first_catalog.default.silver_produtos")
pedidos = spark.table("first_catalog.default.silver_pedidos")

# Selecionar apenas os atributos necessários das dimensões
dim_cliente = clientes.select(
    "id_cliente",
    "segmento",
    "pais",
    "mercado",
    "regiao"
)

dim_produto = produtos.select(
    "id_produto",
    "categoria",
    "subcategoria",
    "nome_produto"
)

dim_pedido = pedidos.select(
    "id_pedido",
    "data_pedido",
    "data_envio",
    "modo_envio",
    "prioridade_pedido",
    "dias_para_envio"
)

# Começar pelas vendas e manter as suas chaves originais
fact_vendas = (
    vendas
    .join(
        dim_cliente,
        vendas.cliente == dim_cliente.id_cliente,
        "left"
    )
    .join(
        dim_produto,
        vendas.produto == dim_produto.id_produto,
        "left"
    )
    .join(
        dim_pedido,
        vendas.pedido == dim_pedido.id_pedido,
        "left"
    )
    .select(
        # Chaves da linha de venda
        vendas.pedido.alias("id_pedido"),
        vendas.cliente.alias("id_cliente"),
        vendas.produto.alias("id_produto"),

        # Métricas
        vendas.valor_venda,
        vendas.quantidade_vendida,
        vendas.custo_envio,

        # Atributos do cliente
        dim_cliente.segmento,
        dim_cliente.pais,
        dim_cliente.mercado,
        dim_cliente.regiao,

        # Atributos do produto
        dim_produto.categoria,
        dim_produto.subcategoria,
        dim_produto.nome_produto,

        # Atributos do pedido
        dim_pedido.data_pedido,
        dim_pedido.data_envio,
        dim_pedido.modo_envio,
        dim_pedido.prioridade_pedido,
        dim_pedido.dias_para_envio
    )
)

# Validar a granularidade e as correspondências
print("Linhas de vendas Silver:", vendas.count())
print("Linhas na fact_vendas:", fact_vendas.count())

print(
    "Vendas sem correspondência de cliente:",
    fact_vendas.filter(F.col("segmento").isNull()).count()
)

print(
    "Vendas sem correspondência de produto:",
    fact_vendas.filter(F.col("categoria").isNull()).count()
)

print(
    "Vendas sem correspondência de pedido:",
    fact_vendas.filter(F.col("data_pedido").isNull()).count()
)

# Guardar a tabela Gold
(
    fact_vendas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("first_catalog.default.fact_vendas")
)

print("Tabela fact_vendas guardada com sucesso.")

display(fact_vendas.limit(10))


Linhas de vendas Silver: 51290
Linhas na fact_vendas: 51290
Vendas sem correspondência de cliente: 0
Vendas sem correspondência de produto: 0
Vendas sem correspondência de pedido: 0
Tabela fact_vendas guardada com sucesso.


id_pedido,id_cliente,id_produto,valor_venda,quantidade_vendida,custo_envio,segmento,pais,mercado,regiao,categoria,subcategoria,nome_produto,data_pedido,data_envio,modo_envio,prioridade_pedido,dias_para_envio
CA-2012-124891,RH-19495,TEC-AC-10003033,2309.650,7,933.570,Consumidor,United States,US,East,Tecnologia,Accessories,Plantronics CS510 - Over-the-Head monaural Wireless Headset System,2012-07-31,2012-07-31,Mesmo Dia,Critico,0
IN-2013-77878,JR-16210,FUR-CH-10003950,3709.395,9,923.630,Corporativo,Australia,APAC,Oceania,Moveis,Chairs,"Novimex Executive Leather Armchair, Black",2013-02-05,2013-02-07,Segunda Classe,Critico,2
IN-2013-71249,CR-12730,TEC-PH-10004664,5175.171,9,915.490,Consumidor,Australia,APAC,Oceania,Tecnologia,Phones,"Nokia Smart Phone, with Caller ID",2013-10-17,2013-10-18,Primeira Classe,Medio,1
ES-2013-1579342,KM-16375,TEC-PH-10004583,2892.510,5,910.160,Outros,Germany,EU,Central,Tecnologia,Phones,"Motorola Smart Phone, Cordless",2013-01-28,2013-01-30,Primeira Classe,Medio,2
SG-2013-4320,RH-9495,TEC-SHA-10000501,2832.960,8,903.040,Consumidor,Senegal,Africa,Africa,Tecnologia,Copiers,"Sharp Wireless Fax, High-Speed",2013-11-05,2013-11-06,Mesmo Dia,Critico,1
IN-2013-42360,JM-15655,TEC-PH-10000030,2862.675,5,897.350,Corporativo,Australia,APAC,Oceania,Tecnologia,Phones,"Samsung Smart Phone, with Caller ID",2013-06-28,2013-07-01,Segunda Classe,Critico,3
IN-2011-81826,TS-21340,FUR-CH-10004050,1822.080,4,894.770,Consumidor,New Zealand,APAC,Oceania,Moveis,Chairs,"Novimex Executive Leather Armchair, Adjustable",2011-11-07,2011-11-09,Primeira Classe,Critico,2
IN-2012-86369,MB-18085,FUR-TA-10002958,5244.840,6,878.380,Consumidor,New Zealand,APAC,Oceania,Moveis,Tables,"Chromcraft Conference Table, Fully Assembled",2012-04-14,2012-04-18,Classe Padrao,Alto,4
CA-2014-135909,JW-15220,OFF-BI-10003527,5083.960,5,867.690,Corporativo,United States,US,West,Material de Escritorio,Binders,Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind,2014-10-14,2014-10-21,Classe Padrao,Baixo,7
CA-2012-116638,JH-15985,FUR-TA-10000198,4297.644,13,865.740,Consumidor,United States,US,South,Moveis,Tables,Chromcraft Bull-Nose Wood Oval Conference Tables & Bases,2012-01-28,2012-01-31,Segunda Classe,Critico,3
